# LlamaIndex com AgentCore Memory - Assistente de Pesquisa Acadêmica (Memória de Curto Prazo)

## Introdução

Este notebook demonstra como integrar as capacidades de Memória do Amazon Bedrock AgentCore com LlamaIndex para criar um Assistente de Pesquisa Acadêmica. Vamos focar na persistência de **memória de curto prazo** dentro de uma única sessão de pesquisa - permitindo que o assistente lembre artigos, descobertas e contexto de pesquisa ao longo de uma conversa.

## Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Memória Conversacional de Curto Prazo                                           |
| Caso de uso do agente | Assistente de Pesquisa Acadêmica                                              |
| Framework Agêntico  | LlamaIndex                                                                       |
| Modelo LLM          | Anthropic Claude 3.7 Sonnet                                                     |
| Componentes do tutorial | Memória de Curto Prazo AgentCore, Agente LlamaIndex, Ferramentas de Pesquisa |
| Complexidade do exemplo | Iniciante                                                                    |

Você aprenderá a:
- Criar AgentCore Memory para persistência de dados de pesquisa
- Usar integração nativa de memória do LlamaIndex
- Construir ferramentas específicas de pesquisa para análise de artigos
- Manter contexto de pesquisa dentro de uma única sessão
- Testar limites de memória e isolamento de sessão

## Contexto do Cenário

Neste exemplo, vamos criar um "Assistente de Pesquisa Acadêmica" que ajuda pesquisadores a rastrear artigos, descobertas e tópicos de pesquisa dentro de uma única sessão de pesquisa. O assistente usa AgentCore Memory para manter contexto sobre artigos revisados, descobertas-chave encontradas e progresso da pesquisa ao longo da conversa.

## Visão Geral da Arquitetura

![Arquitetura LlamaIndex AgentCore Memória de Curto Prazo](LlamaIndex-AgentCore-STM-Arch.png)

## Pré-requisitos

- Python 3.10+
- Conta AWS com permissões apropriadas
- Role IAM AWS com permissões do AgentCore Memory:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Acesso aos modelos do Amazon Bedrock

## Passo 1: Instalar Dependências e Configuração

In [ ]:
# Install necessary libraries
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3

In [ ]:
# Import required components
from bedrock_agentcore.memory import MemoryClient
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

## Passo 2: Configuração do AgentCore Memory

Criar ou obter o recurso de AgentCore Memory para nosso assistente de pesquisa:

In [ ]:
# Create AgentCore Memory resource
region = os.getenv('AWS_REGION', 'us-east-1')
client = MemoryClient(region_name=region)

try:
    response = client.create_memory_and_wait(
        name=f'AcademicResearchShortTerm_{int(datetime.now().timestamp())}',
        description='Academic research assistant short-term memory for single session context',
        strategies=[],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )
    memory_id = response['id']
    print(f"✅ Created AgentCore Memory: {memory_id}")
except Exception as e:
    print(f"❌ Error creating memory: {e}")
    memory_id = "your-memory-id-here"  # Replace with existing memory ID

## Passo 3: Implementação das Ferramentas de Pesquisa

Definir ferramentas especializadas para tarefas de pesquisa acadêmica:

In [ ]:
def save_paper_summary(title: str, authors: str, key_findings: str) -> str:
    """Save a research paper summary with title, authors, and key findings"""
    print(f"📄 Saved paper: {title} by {authors}")
    return f"Successfully saved paper summary for '{title}'"

def track_research_topic(topic: str, status: str) -> str:
    """Track research topic progress with current status"""
    print(f"🔬 Tracking research topic: {topic} (Status: {status})")
    return f"Now tracking research topic: {topic} with status {status}"

def save_research_finding(finding: str, confidence: str) -> str:
    """Save a research finding with confidence level"""
    print(f"💡 Research finding saved with {confidence} confidence")
    return f"Saved research finding with {confidence} confidence level"

# Create tool objects for the agent
research_tools = [
    FunctionTool.from_defaults(fn=save_paper_summary),
    FunctionTool.from_defaults(fn=track_research_topic),
    FunctionTool.from_defaults(fn=save_research_finding)
]

## Passo 4: Implementação do Agente LlamaIndex

Criar o agente assistente de pesquisa com contexto de memória de curto prazo:

In [ ]:
# Configuration for SHORT-TERM memory (single session)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# Create memory context for single session
context = AgentCoreMemoryContext(
    actor_id="academic-researcher",
    memory_id=memory_id,
    session_id="research-session-today",  # Same session throughout
    namespace="/academic-research/"
)

# Initialize AgentCore Memory and LLM
agentcore_memory = AgentCoreMemory(context=context)
llm = BedrockConverse(model=MODEL_ID)

# Create the research assistant agent
research_agent = FunctionAgent(
    tools=research_tools,
    llm=llm,
    verbose=True
)

print("✅ Academic Research Assistant with short-term memory is ready!")

## Passo 5: Testando Capacidades de Memória de Curto Prazo

Vamos testar a memória de curto prazo do nosso assistente de pesquisa através de uma sessão de pesquisa abrangente.

### Teste 1: Inicialização da Sessão

In [ ]:
# Initialize research session with detailed context
response = await research_agent.run(
    "I'm Dr. Sarah Smith from MIT's Computer Science Department, starting research on 'Machine Learning in Healthcare Applications'. "
    "Track this topic with status 'Literature Review'.",
    memory=agentcore_memory
)

print("🎯 Session Initialization:")
print(response)

### Teste 2: Adicionando Artigos de Pesquisa

In [ ]:
# Add first paper with detailed metrics
response = await research_agent.run(
    "Save paper: 'Deep Learning for Medical Image Analysis' by Zhang et al. "
    "Key findings: CNNs achieve 95.2% accuracy in chest X-ray diagnosis, 12% improvement over radiologists, "
    "trained on 100,000 images with 0.03 false positive rate.",
    memory=agentcore_memory
)

print("📄 Paper 1 Added:")
print(response)

In [ ]:
# Add second paper with contrasting findings
response = await research_agent.run(
    "Save paper: 'Transformers in Medical NLP' by Johnson et al. "
    "Key findings: BERT models achieve 89.1% F1-score in clinical note classification, "
    "struggle with rare diseases (<70% accuracy), excel at symptom extraction (94% precision).",
    memory=agentcore_memory
)

print("📄 Paper 2 Added:")
print(response)

### Teste 3: Recuperação de Identidade e Contexto

In [ ]:
# Test identity and research context recall
response = await research_agent.run(
    "What's my name, institution, and current research focus?",
    memory=agentcore_memory
)

print("🧠 Identity Recall Test:")
print(response)
print("\n✅ Expected: Dr. Sarah Smith, MIT, Machine Learning in Healthcare")

### Teste 4: Recuperação de Métricas Detalhadas

In [ ]:
# Test specific metric recall
response = await research_agent.run(
    "What were the exact accuracy percentages mentioned in the papers I reviewed? "
    "Which authors wrote about CNNs vs Transformers?",
    memory=agentcore_memory
)

print("📊 Detailed Metrics Recall:")
print(response)
print("\n✅ Expected: Zhang et al - CNNs 95.2%, Johnson et al - BERT 89.1%")

### Teste 5: Raciocínio Contextual

In [ ]:
# Test contextual understanding and reasoning
response = await research_agent.run(
    "Based on the papers I've reviewed, which approach would be better for analyzing "
    "chest X-rays vs clinical notes? Explain your reasoning.",
    memory=agentcore_memory
)

print("🤔 Contextual Reasoning Test:")
print(response)
print("\n✅ Expected: CNNs for X-rays (Zhang paper), Transformers for clinical notes (Johnson paper)")

### Teste 6: Síntese de Descobertas de Pesquisa

In [ ]:
# Add synthesized research finding
response = await research_agent.run(
    "Based on Zhang's CNN results (95.2% accuracy) and Johnson's Transformer results (89.1% F1-score), "
    "I conclude that deep learning models consistently achieve >85% accuracy in healthcare tasks. "
    "This finding has high confidence. Save it.",
    memory=agentcore_memory
)

print("🔬 Research Finding Synthesis:")
print(response)

### Teste 7: Capacidade de Referência Cruzada

In [ ]:
# Test cross-referencing between findings and papers
response = await research_agent.run(
    "How does my research finding about >85% accuracy relate to the specific results "
    "from Zhang and Johnson? What evidence supports this conclusion?",
    memory=agentcore_memory
)

print("🔗 Cross-Reference Test:")
print(response)
print("\n✅ Expected: Reference to Zhang 95.2% and Johnson 89.1% as supporting evidence")

### Teste 8: Cenário de Aplicação Prática

In [ ]:
# Test practical application of accumulated knowledge
response = await research_agent.run(
    "I'm writing a grant proposal for healthcare AI research. What evidence can I cite "
    "about deep learning effectiveness? Include specific numbers and authors.",
    memory=agentcore_memory
)

print("📝 Grant Proposal Support:")
print(response)
print("\n✅ Expected: Comprehensive summary with Zhang 95.2%, Johnson 89.1%, synthesis finding")

## Passo 6: Testando Limites de Sessão

Vamos testar os limites da memória de curto prazo criando uma sessão diferente:

In [ ]:
# Create a different session context
new_session_context = AgentCoreMemoryContext(
    actor_id="academic-researcher",
    memory_id=memory_id,
    session_id="different-research-session",  # Different session ID
    namespace="/academic-research/"
)

new_session_memory = AgentCoreMemory(context=new_session_context)

# Test memory isolation
response = await research_agent.run(
    "What research have I been working on? What specific accuracy numbers did I find?",
    memory=new_session_memory
)

print("🚧 Session Boundary Test (Different Session):")
print(response)
print("\n✅ Expected: Limited or no recall from previous session (short-term memory boundary)")

In [ ]:
# Return to original session to verify persistence
response = await research_agent.run(
    "Now back in my original session - what were the accuracy numbers from Zhang and Johnson again?",
    memory=agentcore_memory  # Original session memory
)

print("🔄 Original Session Return:")
print(response)
print("\n✅ Expected: Full recall of Zhang 95.2%, Johnson 89.1%")

## 🧪 Validação Automatizada de Testes
Execute estas células para validar que a integração de memória está funcionando corretamente:

In [ ]:
# Define validation functions inline
class TestValidator:
    def __init__(self):
        self.results = {}
    
    def validate_memory_recall(self, response):
        """Check if agent can recall information from earlier in the session"""
        # Check for substantive response (not just "I don't know")
        has_content = len(response) > 50
        # Check for memory indicators
        has_memory_indicators = any(word in response.lower() for word in 
            ['earlier', 'mentioned', 'discussed', 'previously', 'you', 'we', 'our'])
        return "✅ PASS" if (has_content and has_memory_indicators) else "❌ FAIL"
    
    def validate_session_memory(self, response):
        """Check if agent maintains context within session"""
        has_memory_content = len(response) > 100 and any(word in response.lower() for word in 
            ['previous', 'earlier', 'mentioned', 'discussed', 'before', 'already'])
        return "✅ PASS" if has_memory_content else "❌ FAIL"
    
    def validate_cross_reference(self, response):
        """Check if agent can connect current query to previous context"""
        # Look for connecting language
        connecting_words = ['relate', 'connection', 'previous', 'earlier', 'discussed', 
                           'mentioned', 'context', 'based on', 'as we', 'as i']
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "✅ PASS" if (has_connection and has_substance) else "❌ FAIL"
    
    def run_validation_summary(self, test_results):
        print("🧪 COMPREHENSIVE TEST VALIDATION SUMMARY")
        print("=" * 60)
        
        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "PASS" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
        
        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")
        
        print("=" * 60)
        print(f"📊 Overall Pass Rate: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
        
        if pass_rate >= 80:
            print("✅ EXCELLENT: Memory integration working correctly!")
        elif pass_rate >= 60:
            print("⚠️  GOOD: Most memory features working, some issues to investigate")
        else:
            print("❌ NEEDS ATTENTION: Memory integration has significant issues")
        
        return pass_rate

validator = TestValidator()
print("✅ Validation functions loaded!")

In [ ]:
# Run all validation tests
test_results = {}

# Test 1: Memory recall - can the agent recall what was discussed?
response1 = await research_agent.run("What have we discussed so far in this session?", memory=agentcore_memory)
test_results['Memory Recall'] = validator.validate_memory_recall(str(response1))
print(f"Response 1 length: {len(str(response1))} chars")

# Test 2: Session memory - does the agent maintain context?
response2 = await research_agent.run("What did we talk about earlier?", memory=agentcore_memory)
test_results['Session Memory'] = validator.validate_session_memory(str(response2))
print(f"Response 2 length: {len(str(response2))} chars")

# Test 3: Cross-reference capability - can it connect to previous context?
response3 = await research_agent.run("How does this relate to what we discussed before?", memory=agentcore_memory)
test_results['Cross Reference'] = validator.validate_cross_reference(str(response3))
print(f"Response 3 length: {len(str(response3))} chars")

# Display results
validator.run_validation_summary(test_results)

## Resumo

Neste notebook, demonstramos:

✅ **Integração de Memória de Curto Prazo**: Usando AgentCore Memory com LlamaIndex para persistência com escopo de sessão

✅ **Ferramentas Específicas de Pesquisa**: Resumos de artigos, rastreamento de tópicos e armazenamento de descobertas

✅ **Conversas Contextuais**: O assistente lembra informações detalhadas dentro da sessão

✅ **Capacidade de Referência Cruzada**: Conectando descobertas entre múltiplos artigos e interações

✅ **Limites de Sessão**: Isolamento de memória entre diferentes sessões de conversa

✅ **Aplicações Práticas**: Suporte a propostas de financiamento e síntese de pesquisa

O Assistente de Pesquisa Acadêmica demonstra como a memória de curto prazo permite conversas naturais e contextuais dentro de uma única sessão de pesquisa, mantendo limites claros entre diferentes threads de conversa.

## Limpeza

Vamos deletar a memória para limpar os recursos usados neste notebook:

In [ ]:
# Clean up AgentCore Memory resource
try:
    client.delete_memory(memory_id)
    print(f"✅ Successfully deleted memory: {memory_id}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")